# 1. Environment Setup & Dependencies
*Run this cell first to install all necessary system packages and Python libraries.*

In [ ]:
# Install system dependencies
!apt-get install ffmpeg -y

# Install Python libraries
!pip install yt-dlp pydub pandas tqdm noisereduce speechbrain torchaudio openai-whisper openpyxl spleeter --quiet

# 2. Global Configuration & Google Drive
*Mount the Drive and Sets the pipeline toggles*

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# ==========================================
# PIPELINE CONFIGURATION
# ==========================================
# Toggle this to False if you want to skip the heavy emotion detection model
# and default all sentences to 'neutral' to save time.
DETECT_EMOTIONS = False

print(f"Pipeline initialized. Emotion Detection is set to: {DETECT_EMOTIONS}")

# 3. Phase 1: Audio Ingestion (YouTube or Drive)
*Run this to fetch or copy the audio*

In [ ]:
from yt_dlp import YoutubeDL
import os
import shutil

def setup_folder_structure(main_folder, project_name):
    """Creates the standard Main/ and Parts/ folder structure for the pipeline."""
    project_folder = os.path.join(main_folder, project_name)
    main_folder_path = os.path.join(project_folder, 'Main')
    parts_folder_path = os.path.join(project_folder, 'Parts')

    os.makedirs(main_folder_path, exist_ok=True)
    os.makedirs(parts_folder_path, exist_ok=True)

    return main_folder_path, parts_folder_path

def download_from_youtube(main_folder, urls, audio_format):
    if audio_format.lower() not in ["mp3", "wav"]:
        print("❌ Invalid audio format. Please choose either 'mp3' or 'wav'.")
        return

    for url in urls.split(','):
        url = url.strip()
        with YoutubeDL({'quiet': True}) as ydl:
            try:
                info = ydl.extract_info(url, download=False)
                video_title = info.get('title', 'Unknown').replace('/', '_')
                video_id = info.get('id', 'Unknown')
                project_name = f"{video_title}_{video_id}"

                main_folder_path, _ = setup_folder_structure(main_folder, project_name)

                download_options = {
                    'format': 'bestaudio/best',
                    'outtmpl': os.path.join(main_folder_path, f'{video_title}.%(ext)s'),
                    'subtitleslangs': ['ar'],
                    'writesubtitles': True,
                    'postprocessors': [{
                        'key': 'FFmpegExtractAudio',
                        'preferredcodec': audio_format.lower(),
                        'preferredquality': '192',
                    }],
                    'quiet': True,
                }

                print(f"⏳ Downloading: {video_title}...")
                with YoutubeDL(download_options) as downloader:
                    downloader.download([url])
                print(f"✅ Downloaded successfully as {audio_format.upper()}")
            except Exception as e:
                print(f"❌ Failed to download {url}: {str(e)}")

def copy_from_drive(main_folder, file_path):
    if not os.path.exists(file_path):
        print(f"❌ File not found at path: {file_path}")
        return

    # Check if format is supported by Phase 2
    ext = os.path.splitext(file_path)[1].lower()
    if ext not in ['.mp3', '.wav']:
        print(f"❌ Unsupported format '{ext}'. Phase 2 requires .mp3 or .wav")
        return

    # Extract filename to use as the project folder name
    file_name = os.path.basename(file_path)
    project_name = os.path.splitext(file_name)[0].replace('/', '_')

    print(f"⏳ Preparing pipeline structure for: {file_name}...")
    main_folder_path, _ = setup_folder_structure(main_folder, project_name)

    # Copy the file into the Main/ folder
    destination_path = os.path.join(main_folder_path, file_name)
    shutil.copy2(file_path, destination_path)
    print(f"✅ File successfully copied to pipeline directory: {destination_path}")

# ==========================================
# EXECUTION LOGIC
# ==========================================
print("Select your audio input source:")
print("  [1] Download from YouTube")
print("  [2] Select existing file from Google Drive")
choice = input("Enter 1 or 2: ").strip()

main_folder = input("\nEnter the main pipeline workspace folder (e.g., /content/drive/MyDrive/PipelineWorkspace): ").strip()

if choice == '1':
    urls = input("Enter YouTube URLs separated by commas: ").strip()
    audio_format = input("Enter target audio format (mp3 or wav): ").strip()
    download_from_youtube(main_folder, urls, audio_format)

elif choice == '2':
    file_path = input("Enter the absolute Google Drive file path (e.g., /content/drive/MyDrive/audio.wav): ").strip()
    copy_from_drive(main_folder, file_path)

else:
    print("❌ Invalid choice. Please run the cell again and select 1 or 2.")

# 4. Phase 2: Safe Memory-Chunked Noise & Music Removal
*This runs on the main downloaded file before anything else happens.*

In [ ]:
from pydub import AudioSegment
import noisereduce as nr
import numpy as np
from spleeter.separator import Separator
import shutil

def clean_master_audio(input_path, main_folder_path):
    print(f"\n⏳ Starting Safe Noise/Music Removal for: {input_path}")

    # Paths
    temp_dir = os.path.join(main_folder_path, "temp_chunks")
    os.makedirs(temp_dir, exist_ok=True)
    clean_master_path = os.path.join(main_folder_path, "clean_master.wav")

    audio = AudioSegment.from_file(input_path)

    # Chunking into 10-minute segments to prevent Colab RAM crashes
    chunk_length_ms = 10 * 60 * 1000
    chunks = [audio[i:i + chunk_length_ms] for i in range(0, len(audio), chunk_length_ms)]

    separator = Separator('spleeter:2stems')
    cleaned_chunks = []

    for idx, chunk in enumerate(chunks):
        print(f"   🧹 Cleaning chunk {idx + 1}/{len(chunks)}...")

        # 1. Noisereduce
        samples = np.array(chunk.get_array_of_samples())
        reduced_samples = nr.reduce_noise(y=samples, sr=chunk.frame_rate)
        nr_audio = chunk._spawn(reduced_samples.astype(np.int16).tobytes())

        # Save temp file for Spleeter
        temp_chunk_path = os.path.join(temp_dir, f"chunk_{idx}.wav")
        nr_audio.set_channels(2).set_frame_rate(22050).export(temp_chunk_path, format="wav")

        # 2. Spleeter (Separate vocals from music)
        separator.separate_to_file(temp_chunk_path, temp_dir)

        # Spleeter creates a folder with the vocals. Load those vocals back.
        vocal_path = os.path.join(temp_dir, f"chunk_{idx}", "vocals.wav")
        if os.path.exists(vocal_path):
            cleaned_chunks.append(AudioSegment.from_wav(vocal_path))
        else:
            cleaned_chunks.append(AudioSegment.from_wav(temp_chunk_path)) # Fallback

    # Stitch chunks back together
    print("⏳ Recombining cleaned chunks...")
    final_clean_audio = cleaned_chunks[0]
    for chunk in cleaned_chunks[1:]:
        final_clean_audio += chunk

    final_clean_audio.export(clean_master_path, format="wav", bitrate="16k", codec="pcm_s16le")

    # Cleanup temp files
    shutil.rmtree(temp_dir)
    print(f"✅ Clean Master Audio saved to: {clean_master_path}")
    return clean_master_path

# 5. Phase 3: Whisper Transcription & Audio Segmentation

In [ ]:
import whisper
import pandas as pd
import csv
from datetime import datetime, timedelta
from tqdm import tqdm

model = whisper.load_model("large")
parent_dir = main_folder # From your ingest step

for sub_dir in os.listdir(parent_dir):
    sub_dir_path = os.path.join(parent_dir, sub_dir)
    if not os.path.isdir(sub_dir_path): continue

    main_folder_path = os.path.join(sub_dir_path, "Main")
    if not os.path.exists(main_folder_path): continue

    mp3_files = [f for f in os.listdir(main_folder_path) if f.endswith(".mp3") or f.endswith(".wav")]
    if not mp3_files: continue

    raw_audio_file = os.path.join(main_folder_path, mp3_files[0])

    # --- RUN NEW CLEANING LOGIC HERE ---
    clean_audio_file = clean_master_audio(raw_audio_file, main_folder_path)
    # -----------------------------------

    main_path = os.path.join(sub_dir_path, "")
    csv_file = os.path.join(main_path, "sentence_timestamps.csv")
    output_excel_path = os.path.join(main_path, "final_transcriptions_mapping.xlsx")
    output_dir = os.path.join(main_path, "Parts/")
    os.makedirs(output_dir, exist_ok=True)

    print(f"🎙️ Transcribing cleaned file...")
    result = model.transcribe(clean_audio_file, word_timestamps=True, verbose=False, language="ar")

    # (Keep your exact CSV saving and Audio slicing logic here.
    # Just make sure AudioSegment.from_file() loads the `clean_audio_file`)